# 23-09 · Skanujemy prawdziwy katalog

Praktyka do sekcji [„Skanujemy katalog”](/pl/chapters/rozdzial-23/23-08-skaniruem-katalog.html). Używa prawdziwego pakietu `safesort` (`projects/python/safesort/`).

## Reproducible local environment

```bash
git clone https://github.com/Cartesian-School/safesort.git
cd safesort
python3.14 -m venv .venv
source .venv/bin/activate
# Windows PowerShell: .venv\Scripts\Activate.ps1
python -m pip install -U pip
python -m pip install -e ".[dev]"
python -m pip install jupyter ipykernel
python -m ipykernel install --user --name safesort-py314 --display-name "SafeSort Python 3.14"
jupyter lab
```

Select the **SafeSort Python 3.14** kernel. The diagnostic cell below must
point into this `.venv` and the cloned `src/safesort` tree.

In [ ]:
import sys
import safesort

print(sys.executable)
print(safesort.__file__)

## Cel

Uruchom tę funkcję `safesort.scanner.scan()` tymczasowy katalog z dołączonymi plikami i upewnij się, że znalazł dokładnie to, czego potrzebuje — i nic dodatkowego.

## Example

In [ ]:
import tempfile
from pathlib import Path

from safesort.config import Config
from safesort.scanner import scan

tmpdir = tempfile.TemporaryDirectory()
koren = Path(tmpdir.name)

(koren / "podkatalog").mkdir()
(koren / "podkatalog" / "otchet.pdf").write_text("...", encoding="utf-8")
(koren / "photo.jpg").write_text("...", encoding="utf-8")
(koren / ".git").mkdir()
(koren / ".git" / "config").write_text("...", encoding="utf-8")

fajly = scan(koren, Config())
imena = {f.path.name for f in fajly}
print("Найдено файлов:", len(fajly))
print(imena)

## Sprawdzenie wyniku

In [ ]:
assert imena == {"otchet.pdf", "photo.jpg"}
assert all(f.path.name != "config" for f in fajly)  # .git исключён по умолчанию
print("Верно: сканер нашёл вложенный файл и файл в корне, но не заглянул в .git.")

## Eksperyment — ponowne uruchomienie nie znajduje już posortowanych plików

In [ ]:
(koren / "Sorted" / "documents").mkdir(parents=True)
(koren / "Sorted" / "documents" / "staryj.pdf").write_text("...", encoding="utf-8")

fajly_posle = scan(koren, Config())
imena_posle = {f.path.name for f in fajly_posle}

assert "staryj.pdf" not in imena_posle
assert imena_posle == {"otchet.pdf", "photo.jpg"}
print("Верно: каталог результата Sorted/ исключён из повторного сканирования.")

## Starter

Wypełnij zaznaczone miejsce. Niezmieniony starter nie przechodzi tests.

In [ ]:
def proverit_propusk_ssylki(root: Path):
    # TODO: create photo.jpg and a symlink, call scan(), inspect names.
    raise NotImplementedError


## Task

Write `proverit_propusk_ssylki(root)`: Stwórz target i symlink, potem wróć `True`jeśli scan przegapił link. Jeśli nie ma wsparcia, wróć `None`.

## Tests

Run After task cell: jest podstawowy przykład i przynajmniej jeden skrajny przypadek.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    rezultat_ssylki = proverit_propusk_ssylki(Path(tmp))
assert rezultat_ssylki in (True, None)

# Edge case: an empty directory has no scan results.
with tempfile.TemporaryDirectory() as tmp:
    assert scan(Path(tmp), Config()) == []
print("Tests passed")

## Hint

Przechwycić `(OSError, NotImplementedError)` tylko w pobliżu `symlink_to()`.

## Solution

Pokaż rozwiązanie po własnej próbie</summary>

```python
def proverit_propusk_ssylki(root: Path):
    target = root / "photo.jpg"
    target.write_text("photo", encoding="utf-8")
    link = root / "ssylka_na_foto.jpg"
    try:
        link.symlink_to(target)
    except (OSError, NotImplementedError):
        return None

    names = {file.path.name for file in scan(root, Config())}
    return "photo.jpg" in names and "ssylka_na_foto.jpg" not in names
```

</details>